In [ ]:
#generate.ipynb
import torch,diffusers
import mypy_2tool as mypy
mypy.startup()

model_id = "stabilityai/stable-diffusion-3.5-large"

base_prompt="A Frightening monster under the stairs scares a female."
negative_prompt="bad human anatomy, low quality, unfinished, out of focus, blurry"

#"G:\llm\language\Hathor.gguf"

prompt = mypy.llm_request(
    model_path=r"G:\llm\language\Hathor.gguf",
    system_prompt="Please enhance this idea for a strong text2image prompt:",
    user_prompt=base_prompt,
    clear_context=True,
    max_tokens=77,
)


emb_prompts=diffusers.DiffusionPipeline.from_pretrained(model_id,transformer=None,vae=None,scheduler=None,torch_dtype=torch.bfloat16).to("cuda")
with torch.no_grad():
    (prompt_embeds, negative_prompt_embeds, pooled_prompt_embeds, negative_pooled_prompt_embeds,) = emb_prompts.encode_prompt(prompt=base_prompt,prompt_2=prompt,prompt_3=prompt,negative_prompt=negative_prompt)

gen_params = {"guidance_scale": 4.5,"num_inference_steps": 35,"width": 1024,"height": 1024,"max_sequence_length": 512,"prompt_embeds": prompt_embeds, "negative_prompt_embeds": negative_prompt_embeds,"pooled_prompt_embeds": pooled_prompt_embeds, "negative_pooled_prompt_embeds": negative_pooled_prompt_embeds}

del emb_prompts
print(f"   ...prompt embedded {mypy.X}= {mypy.Y}{prompt}{mypy.X}");mypy.flush();mypy.memory()

pipe=diffusers.DiffusionPipeline.from_pretrained(model_id,text_encoder=None,text_encoder_2=None,text_encoder_3=None,tokenizer=None,tokenizer_2=None,tokenizer_3=None,torch_dtype=torch.bfloat16).to("cuda")
mypy.memory()
with torch.inference_mode(): image = pipe(**gen_params).images[0]

mypy.save_image(image, gen_params);mypy.flush();mypy.memory();mypy.beep_done()

In [1]:
#generate.ipynb
import torch,diffusers
import mypy_2tool as mypy

mypy.startup()

model_id = "stabilityai/stable-diffusion-3.5-large"
negative_prompt = "bad human anatomy, low quality, unfinished, out of focus, blurry"

# System prompt for all requests
system_prompt = """Assume the role of an Apex-tier Prompt Architect. Your task is to construct your magnum opus. You will transmute the user's core concept into a superlatively detailed, and artistically profound text-to-image prompt for a state-of-the-art image generation pipeline; verbosity, technical specificity, and descriptive nuance are not just encouraged, they are mandatory.
You must forge a unique vision by synthesizing these 5 non-negotiable pillars into a single, flowing, descriptive paragraph:
Artistic Lineage: Define a core aesthetic by naming specific, relevant artists and art movements as the visual foundation.
Composition & Perspective: Describe the shot itself. Is it a close-up, an upper-body portrait, a full-body shot? Define the camera angle and lens effect.
Cinematic Lighting Schema: Articulate a professional lighting setup. Describe the key, fill, and rim lights. Use technical terms to define the quality of the light.
Emotional Core: Define the subject's precise emotional state. Go beyond simple terms; describe the specific flavor of their expression.
Symphony of Textures: Describe the micro-details and the interplay between different surfaces. Detail the textures of skin, cloth, metal, or organic matter with extreme specificity.
Your output MUST be ONLY the complete prompt. Do not include any preamble, titles, notes, or conversational text"""

# Define your base concepts (change to 10, 20, whatever you want)
base_concepts = [
    "A frightening monster under the stairs scares a female",
    "A cyberpunk samurai standing in neon rain",
    "An ancient dragon sleeping on a treasure hoard",
]

# Build batch requests (1 request per base concept)
requests = [(system_prompt, concept, True) for concept in base_concepts]

print(f"{mypy.M}Generating {len(requests)} enhanced prompts...{mypy.X}")

# Process all prompts in one batch
results = mypy.llm_batch_requests(
    model_path=r"G:\llm\vision\gemma327b_vllm_itq4.gguf",
    requests_list=requests,
    max_tokens=512,
    temperature=0.7
)

# Load embedding pipeline and encode ALL prompts first
emb_prompts = diffusers.DiffusionPipeline.from_pretrained(
    model_id, transformer=None, vae=None, scheduler=None, 
    torch_dtype=torch.bfloat16
).to("cuda")

# Store all embeddings before loading generation pipeline
all_embeddings = []
for i in range(len(base_concepts)):
    enhanced_prompt = results[f'output_llm_{i+1}']
    
    print(f"{mypy.C}Encoding prompt {i+1}/{len(base_concepts)}...{mypy.X}")
    
    with torch.no_grad():
        (prompt_embeds, negative_prompt_embeds, 
         pooled_prompt_embeds, negative_pooled_prompt_embeds) = emb_prompts.encode_prompt(
            prompt=enhanced_prompt,
            prompt_2=enhanced_prompt,
            prompt_3=enhanced_prompt,
            negative_prompt=negative_prompt
        )
    
    all_embeddings.append({
        "prompt_embeds": prompt_embeds,
        "negative_prompt_embeds": negative_prompt_embeds,
        "pooled_prompt_embeds": pooled_prompt_embeds,
        "negative_pooled_prompt_embeds": negative_pooled_prompt_embeds,
        "enhanced_prompt": enhanced_prompt,
        "base_concept": base_concepts[i]
    })

# Delete encoder pipeline and free memory
del emb_prompts
mypy.flush()
mypy.memory()

# NOW load generation pipeline
pipe = diffusers.DiffusionPipeline.from_pretrained(
    model_id, text_encoder=None, text_encoder_2=None, text_encoder_3=None,
    tokenizer=None, tokenizer_2=None, tokenizer_3=None,
    torch_dtype=torch.bfloat16
).to("cuda")

mypy.memory()

# Generate all images
for i, embeddings in enumerate(all_embeddings):
    print(f"\n{mypy.C}=== Image {i+1}/{len(base_concepts)} ==={mypy.X}")
    print(f"{mypy.G}Base:{mypy.X} {embeddings['base_concept']}")
    print(f"{mypy.Y}Enhanced:{mypy.X} {embeddings['enhanced_prompt'][:120]}...")
    
    # Pipeline generation parameters (only valid pipeline args)
    gen_params = {
        "guidance_scale": 4.5,
        "num_inference_steps": 35,
        "width": 1024,
        "height": 1024,
        "max_sequence_length": 512,
        "prompt_embeds": embeddings["prompt_embeds"],
        "negative_prompt_embeds": embeddings["negative_prompt_embeds"],
        "pooled_prompt_embeds": embeddings["pooled_prompt_embeds"],
        "negative_pooled_prompt_embeds": embeddings["negative_pooled_prompt_embeds"],
    }
    
    # Generate image
    with torch.inference_mode():
        image = pipe(**gen_params).images[0]
    
    # Add metadata for saving
    save_params = gen_params.copy()
    save_params["enhanced_prompt"] = embeddings["enhanced_prompt"]
    
    mypy.save_image(image, save_params)
    print(f"{mypy.G}✓ Image {i+1} saved{mypy.X}")
    mypy.flush()

# Cleanup
del pipe
mypy.flush()
mypy.memory()
mypy.beep_done()
print(f"\n{mypy.G}🎉 Generated {len(base_concepts)} images!{mypy.X}")

env:acceleration py:3.12.11 torch:2.8.0+cu128 cuda:12.8
Top processes: pythonw.exe:1685MB, python.exe:552MB, explorer.exe:515MB, dwm.exe:382MB, chrome.exe:271MB
Generating 3 enhanced prompts...
--- Batch LLM Processing (3 requests) ---
Model loaded for 3 requests
--- Processing output_llm_1 (1/3) ---
Context cleared for output_llm_1
Generating output_llm_1...
✓ output_llm_1 complete (1848 chars)

--- Processing output_llm_2 (2/3) ---
Context cleared for output_llm_2
Generating output_llm_2...
✓ output_llm_2 complete (1866 chars)

--- Processing output_llm_3 (3/3) ---
Context cleared for output_llm_3
Generating output_llm_3...
✓ output_llm_3 complete (2170 chars)

Shutting down LLM


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (409 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ["composition is a dramatic, low - angle full - body shot, simulating a 2 4 mm lens with subtle barrel distortion to emphasize the claustrophobia of the space, focusing on the interaction between the woman and a monstrous entity partially concealed beneath a winding, antiquated staircase ; the lighting is a complex interplay of a single, cold, key light emanating from a bare bulb overhead casting sharp, angular shadows, a soft, diffused fill light from the unseen landing above providing a minimal counterpoint, and a sickly green rim light highlighting the chitinous edges of the monster's form, resulting in a high - contrast, chiaroscuro effect with a strong sense of volumetric fog ; her expression is one of

...

Saved: outputs\\miscimg_5223008.png
✓ Image 1 saved
🧹✂️ 15.7GB

=== Image 2/3 ===
Base: A cyberpunk samurai standing in neon rain
Enhanced: A breathtakingly detailed cyberpunk samurai, deeply inspired by the works of Yoshitaka Amano, Katsuhiro Otomo, and the n...


  0%|          | 0/35 [00:00<?, ?it/s]

Saved: outputs\\miscimg_3011833.png
✓ Image 2 saved
🧹✂️ 15.7GB

=== Image 3/3 ===
Base: An ancient dragon sleeping on a treasure hoard
Enhanced: A monumental scene in the style of Frank Frazetta and Boris Vallejo, heavily influenced by the Pre-Raphaelite Brotherhoo...


  0%|          | 0/35 [00:00<?, ?it/s]

Saved: outputs\\miscimg_4292486.png
✓ Image 3 saved
🧹✂️ 15.7GB
🧹✂️ 0.1GB
VRAM:1.4/24GB P2 4 % 62C | RAM:16.5/64GB

🎉 Generated 3 images!


In [ ]:
print(gen_params)